# Bài 14 · Kể chuyện bằng dữ liệu & thẩm định phân tích AI

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu buổi học** — sau notebook này, bạn:

1. Thực thi **quy trình thẩm định (audit) 4 bước** (truy số → phương pháp → diễn giải → phán quyết)
   trên một báo cáo do AI viết.
2. Viết lại kết luận sai thành kết luận đúng mực — lời vừa với số.
3. Đóng gói kết quả thành **bảng phán quyết** + đoạn tóm tắt kiểu kim tự tháp.

## Hồ sơ vụ việc: "Báo cáo AI" về thị trường Airbnb Santiago

Đoạn dưới là báo cáo do một chatbot viết khi được đưa dữ liệu Santiago (snapshot 29/06/2026)
và prompt *"viết báo cáo thị trường ngắn, ấn tượng"*. **Nhiệm vụ: thẩm định cả 5 kết luận.**

---

> ### Santiago Airbnb Market Report — 06/2026
>
> **KL1.** Thị trường có **18.534 listing**, trong đó **81% là nguyên căn** — nguồn cung
> nghiêng hẳn về cho thuê cả nhà.
>
> **KL2.** Giá thuê **trung bình 118.200 CLP/đêm** (~3,3 triệu VND) — du khách nên chuẩn bị
> ngân sách tương ứng.
>
> **KL3.** Số review **tháng 6/2026 giảm 27%** so với tháng 5 — thị trường đang
> **hạ nhiệt đáng lo ngại**.
>
> **KL4.** Phòng có tên nhắc đến metro **rẻ hơn ~11%** — cho thấy **vị trí gần metro làm giảm
> giá trị** bất động sản cho thuê.
>
> **KL5.** Trong các quận có ≥500 listing, **Lo Barnechea đắt nhất** với giá trung vị
> **426.230 CLP/đêm**, bỏ xa quận thứ hai (Las Condes, ~97.000).

---

Quy trình cho từng kết luận: **(1) truy số** — tự tính lại; **(2) kiểm phương pháp** —
mean/median? mùa vụ? QA?; **(3) kiểm diễn giải** — lời có vừa với số?; **(4) phán quyết**.

In [ ]:
import pandas as pd
import numpy as np

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29"
df = pd.read_csv(f"{BASE}/visualisations/listings.csv")
rv = pd.read_csv(f"{BASE}/visualisations/reviews.csv", parse_dates=["date"])
print(df.shape, rv.shape)

## KL1 — "18.534 listing, 81% nguyên căn"

In [ ]:
# Bước 1: truy số
so_listing = len(df)
ty_le_nguyen_can = (df["room_type"] == "Entire home/apt").mean()
print(f"Số listing: {so_listing:,} | nguyên căn: {ty_le_nguyen_can:.1%}")

**Phán quyết KL1:** ✅ số đúng (18.534; 81,0%), phương pháp hợp lệ, lời vừa với số. **Giữ nguyên.**
Thẩm định không phải bới lỗi bằng mọi giá — xác nhận đúng cũng là kết quả.

## KL2 — "trung bình 118.200 CLP/đêm, du khách chuẩn bị ngân sách tương ứng"

In [ ]:
# Bước 1: truy số — mean có đúng 118.200 không?
gia = df.loc[df["price"] > 0, "price"]
print(f"mean   : {df['price'].mean():,.0f}")
print(f"median : {df['price'].median():,.0f}")
print(f"P99    : {gia.quantile(0.99):,.0f}  (max: {gia.max():,.0f})")

In [ ]:
# Bước 2: kiểm phương pháp — mean nhạy với outlier cỡ nào?
mean_sach = gia[gia <= gia.quantile(0.99)].mean()
print(f"mean sau khi bỏ 1% đuôi: {mean_sach:,.0f}  (mean gốc phồng {df['price'].mean()/mean_sach - 1:.0%})")

**Phán quyết KL2:** số đúng ✓ nhưng phương pháp sai cho mục đích "ngân sách du khách" —
mean bị 1% outlier thổi phồng ~44%; nửa thị trường nằm dưới 59.000 CLP.

✍️ **Viết lại cho đúng mực:** *"Một nửa số phòng có giá dưới 59.000 CLP/đêm (~1,7 triệu VND);
mức trung bình 118.200 bị một nhóm nhỏ listing giá bất thường kéo lên, không phản ánh
ngân sách điển hình."*

## KL3 — "tháng 6 giảm 27% so với tháng 5 → hạ nhiệt đáng lo ngại"

In [ ]:
# Bước 1: truy số — có đúng -27%?
thang = rv[rv["date"] < "2026-07-01"].set_index("date").resample("ME").size()
mom = thang.pct_change().iloc[-1]
print(f"Tháng 6 vs tháng 5: {mom:.1%}")

# So cùng kỳ (buổi 8) kể chuyện khác hẳn:
yoy = thang.iloc[-1] / thang.iloc[-13] - 1
print(f"Tháng 6/2026 vs tháng 6/2025: {yoy:+.1%}")

# Nhưng khoan — tháng 6 các năm TRƯỚC có giảm so với tháng 5 không?
for nam in [2023, 2024, 2025]:
    t5 = thang[f"{nam}-05"].iloc[0]; t6 = thang[f"{nam}-06"].iloc[0]
    print(f"{nam}: T6 vs T5 = {t6/t5-1:+.1%}")

Lạ: các năm trước tháng 6 **không** giảm. Vậy −27% năm nay là thật? Trước khi kết luận,
kiểm tra **chính cái thước đo**: snapshot chụp ngày 29/06 — khách ở cuối tháng 6 *chưa kịp
viết review* thì bản chụp đã đóng máy. Tháng cuối của mọi snapshot đều **hụt một cách hệ thống**.

Chứng minh bằng dữ liệu — so cùng một tháng (9/2025) nhìn từ hai snapshot khác nhau:

In [ ]:
# Bằng chứng right-censoring: tháng 9/2025 nhìn từ snapshot 09/2025 vs snapshot 06/2026
r9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/reviews.csv", parse_dates=["date"])
for nhan, r in [("snapshot 09/2025 (vừa chụp xong)", r9), ("snapshot 06/2026 (9 tháng sau)", rv)]:
    m = r.set_index("date").resample("ME").size()
    print(f"{nhan:35} T9/2025 = {m.get(pd.Timestamp('2025-09-30'), 0):,}")

Cùng tháng 9/2025: bản chụp nóng hổi ghi **14.264**, chín tháng sau con số "mọc" thành
**17.301** (+21%) — vì review viết trễ dần dần nhập kho. (Để ý chiều ngược: một số tháng cũ
lại *teo đi* giữa hai snapshot — listing rời sàn mang theo review của nó. Bảng `reviews` là
một sinh vật sống!)

**Phán quyết KL3:** số −27% đúng ✓ nhưng tháng cuối snapshot **luôn hụt hệ thống**, nên mức giảm
thật không đo được từ bản chụp này; so cùng kỳ vẫn +7%. Kết luận "hạ nhiệt đáng lo ngại" →
**không kiểm chứng được từ một snapshot**.

✍️ **Viết lại:** *"Số review tháng gần nhất chưa phản ánh đủ (khách chưa kịp viết); so với
cùng kỳ 2025 thị trường vẫn +7%. Cần snapshot kế tiếp để đánh giá xu hướng quý 2/2026."*

## KL4 — "gần metro làm giảm giá trị bất động sản"

In [ ]:
# Bước 1: truy số
gan_metro = df["name"].str.contains("metro", case=False, na=False)
chenh = df.loc[gan_metro, "price"].median() / df.loc[~gan_metro, "price"].median() - 1
print(f"Chênh lệch trung vị: {chenh:+.1%}")

# Bước 3: kiểm diễn giải — biến ẩn "loại phòng"?
print()
print(pd.crosstab(gan_metro, df["room_type"], normalize="index").round(2))

In [ ]:
# So sánh TRONG TỪNG loại phòng — chênh lệch còn bao nhiêu?
so_trong_nhom = (df.groupby([df["room_type"], gan_metro])["price"].median()
                   .unstack())
so_trong_nhom["chênh %"] = (so_trong_nhom[True] / so_trong_nhom[False] - 1) * 100
so_trong_nhom.round(1)

Kết quả kiểm confounder gây bất ngờ hai lần: nhóm "khoe metro" đúng là thiên về phòng riêng,
**nhưng** so trong từng loại phòng thì chênh lệch *vẫn còn ~17%* — biến ẩn "loại phòng"
không giải thích hết.

**Phán quyết KL4:** hiện tượng có thật và bền qua một lần kiểm soát ✓ — nhưng kết luận nhân quả
vẫn **sai logic**: (a) "tên nhắc metro" là *tín hiệu marketing tự chọn* — phòng thiếu lợi thế
khác mới phải khoe metro, đây không phải thước đo khoảng cách; (b) còn vô số biến ẩn khác
(chất lượng nội thất, tầng, tuổi listing…). Dữ liệu quan sát cho phép nói "đi kèm",
không cho phép nói "làm giảm".

✍️ **Viết lại:** *"Listing khoe 'metro' trong tên rẻ hơn ~11–17% kể cả trong cùng loại phòng —
một tín hiệu phân khúc thú vị; xác định tác động nhân quả của vị trí đòi hỏi đo khoảng cách
thật đến ga và thiết kế phân tích khác."*

## KL5 — "Lo Barnechea đắt nhất trong các quận ≥500 listing, 426.230 CLP"

In [ ]:
# Bước 1+2: truy số với đúng điều kiện đề bài (n >= 500)
tk = df.groupby("neighbourhood")["price"].agg(trung_vi="median", n="size")
tk[tk["n"] >= 500].nlargest(3, "trung_vi").round(0)

**Phán quyết KL5:** ✅ đúng cả số lẫn điều kiện lọc. **Giữ nguyên** — có thể bổ sung n=824
để người đọc tự cân nhắc độ tin.

## Tổng kết cuộc thẩm định

In [ ]:
verdict = pd.DataFrame([
    ["KL1: 18.534 listing, 81% nguyên căn", "✓", "✓", "✓", "GIỮ NGUYÊN"],
    ["KL2: mean 118.200 ~ ngân sách",       "✓", "✗ outlier", "✗", "SỬA: dùng median"],
    ["KL3: -27% MoM ~ hạ nhiệt",            "✓", "✗ đuôi snapshot hụt", "✗", "KHÔNG KIỂM ĐƯỢC từ 1 snapshot"],
    ["KL4: metro làm giảm giá",             "✓", "△ đo proxy", "✗ nhân quả", "HẠ CẤP: 'đi kèm'"],
    ["KL5: Lo Barnechea 426k",              "✓", "✓", "✓", "GIỮ NGUYÊN (+n)"],
], columns=["Kết luận", "Số", "Phương pháp", "Diễn giải", "Phán quyết"])
verdict

Nhìn cột "Số": **AI đúng phép tính cả 5 lần**. Lỗi nằm ở phương pháp và diễn giải — đúng vùng
mà môn học này rèn cho bạn. Đó là lý do thẩm định là kỹ năng của *người hiểu dữ liệu*, không phải
của máy tính.

## Bài tập tại lớp

### Bài 1 — Viết đoạn mở đầu kim tự tháp

Từ bảng phán quyết, viết **5 câu** mở đầu báo cáo thẩm định theo cấu trúc: câu trả lời chung →
2 bằng chứng đắt nhất → giới hạn. (Viết vào cell Markdown dưới. Gợi ý câu đầu: *"Ba trong năm
kết luận của báo cáo AI cần sửa hoặc đảo ngược, dù mọi con số đều tính đúng."*)

*(Đoạn của bạn — double-click để viết)*

### Bài 2 — Bẫy mốc so sánh

Tự tạo một "kết luận giật gân nhưng đúng số" từ dữ liệu review: chọn mốc so sánh có lợi
(gợi ý: so với đáy COVID 2020 hoặc một tháng thấp điểm). Rồi viết phiên bản trung thực của
chính kết luận đó. Cảm nhận sự khác biệt — và nhớ cảm giác này khi viết báo cáo bài tập lớn.

In [ ]:
# TODO Bài 2 (scaffold):
t_2020 = thang.loc["2020"].min()
t_moi = thang.iloc[-1]
print(f"Giật gân : 'Review tăng {t_moi/t_2020:.0f} LẦN so với 2020!'")

t_2019 = thang.loc["2019"].mean()
print(f"Trung thực: 'Review gấp ~{t_moi/t_2019:.1f} lần mức trước dịch (TB 2019).'")

### Bài 3 — Simpson mini

Kiểm tra xem nghịch lý Simpson có *thật sự* xảy ra giữa 2 snapshot Santiago không:
tính giá trung vị **toàn thị trường** và **theo từng room_type** cho 09/2025 vs 06/2026.
Chiều nào ngược chiều nào? Tỷ trọng các loại phòng đổi ra sao?

In [ ]:
# TODO Bài 3:
t9 = pd.read_csv("https://data.insideairbnb.com/chile/rm/santiago/"
                 "2025-09-27/visualisations/listings.csv")
for ten, d in [("2025-09", t9), ("2026-06", df)]:
    print(ten, "| chung:", f"{d['price'].median():,.0f}",
          "| mix nguyên căn:", f"{(d['room_type']=='Entire home/apt').mean():.1%}")
print()
print((pd.concat([t9.assign(s="2025-09"), df.assign(s="2026-06")])
       .groupby(["s", "room_type"])["price"].median().unstack().round(0)))

## Thử thách về nhà 🏆 — Thẩm định chính nhóm mình

1. Lấy phần báo cáo bài tập lớn nhóm bạn đã viết (hoặc nhờ một chatbot viết 5 kết luận từ bảng KPI
   của nhóm).
2. Chạy thẩm định 4 bước cho **từng kết luận**; lập bảng phán quyết như trên.
3. Kết luận nào phải sửa? Cập nhật báo cáo + ghi vào `AI_USAGE.md` mục "AI sai ở đâu"
   (nếu lỗi do AI) — nội dung này dùng trực tiếp cho vấn đáp buổi 15.

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| Kim tự tháp: kết luận trước, hành trình xuống phụ lục | Người đọc bận — và rubric chấm điều này |
| Thẩm định 4 bước: truy số → phương pháp → diễn giải → phán quyết | Kỹ năng lõi thời AI; đúng format vấn đáp |
| AI đúng phép tính, sai phương pháp & diễn giải | Biết nhìn CHỖ NÀO khi kiểm tra |
| Mốc so sánh, mùa vụ, mix mẫu, lời quá cỡ | Bốn bẫy tự kiểm trước khi nộp |

**Toàn bộ nội dung môn học kết thúc ở đây.** Buổi 15: vấn đáp bài tập lớn — xem deck hướng dẫn
+ checklist nộp bài. Hẹn gặp! 🏁